# TSL-51 Model Evaluation

Evaluate trained Thai Sign Language recognition models.

## 1. Setup (Run 01_setup.ipynb first)

In [ ]:
import os
import sys
import torch
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/TSL'
os.chdir(WORK_DIR)
sys.path.insert(0, os.path.join(WORK_DIR, 'src'))

print(f'Working directory: {os.getcwd()}')

## 2. Load Trained Model

In [ ]:
# Specify model path (update with your trained model)
MODEL_PATH = '/content/drive/MyDrive/TSL/models/tsl51_gru_20260519_214700.pt'

checkpoint = torch.load(MODEL_PATH, map_location='cpu')
config = checkpoint['config']
classes = checkpoint['classes']
mean = checkpoint.get('mean', 0)
std = checkpoint.get('std', 1)

print(f'Model loaded from: {MODEL_PATH}')
print(f'Config: {config}')
print(f'Number of classes: {len(classes)}')

## 3. Load Test Dataset

In [ ]:
from src.data.loader import load_tsl51_user_sign, load_tsl51_expert, load_tsl51_combined, load_tsl51_expert_full
from sklearn.model_selection import train_test_split

# Map dataset names to loader functions
dataset_loaders = {
    'tsl51_user_sign': load_tsl51_user_sign,
    'tsl51_expert': lambda: load_tsl51_expert(include_augmented=False),
    'tsl51_expert_full': load_tsl51_expert_full,
    'tsl51_combined': load_tsl51_combined
}

print(f'Loading dataset: {config["dataset"]}')
loader_func = dataset_loaders.get(config['dataset'], load_tsl51_user_sign)
X, y, _ = loader_func()

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=config.get('test_split', 0.2), 
    random_state=42,
    stratify=y
)

print(f'Train samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

## 4. Rebuild Model

In [ ]:
from src.core.models import GRUModel, MLPModel

input_dim = X_test.shape[1]
num_classes = len(classes)

if config['model_type'] == 'gru':
    model = GRUModel(
        input_dim=input_dim,
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_layers'],
        num_classes=num_classes,
        dropout=config['dropout']
    )
elif config['model_type'] == 'mlp':
    model = MLPModel(
        input_dim=input_dim,
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_layers'],
        num_classes=num_classes,
        dropout=config['dropout']
    )

model.load_state_dict(checkpoint['state_dict'])
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f'Model rebuilt and loaded on {device}')

## 5. Evaluate on Test Set

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Normalize test data
X_test_normalized = (X_test - mean) / std

# Create dataloader
test_dataset = TensorDataset(
    torch.FloatTensor(X_test_normalized),
    torch.LongTensor(y_test)
)
test_loader = DataLoader(
    test_dataset,
    batch_size=config['batch_size'],
    shuffle=False
)

# Evaluate
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        outputs = model(batch_x)
        preds = outputs.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = (all_preds == all_labels).mean()
print(f'Test accuracy: {accuracy:.4f}')

## 6. Classification Report

In [ ]:
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=classes, zero_division=0))

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

fig_path = os.path.join(WORK_DIR, 'results', 'confusion_matrix.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Confusion matrix saved to: {fig_path}')

## 8. Per-Class Accuracy

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(12, 6))
plt.bar(range(len(classes)), per_class_acc)
plt.xlabel('Class')
plt.ylabel('Accuracy')
plt.title('Per-Class Accuracy')
plt.xticks(range(len(classes)), classes, rotation=45, ha='right')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(WORK_DIR, 'results', 'per_class_accuracy.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Per-class accuracy plot saved to: {fig_path}')
print('\nPer-class accuracies:')
for i, (cls, acc) in enumerate(zip(classes, per_class_acc)):
    print(f'{cls}: {acc:.4f}')

## 9. Save Evaluation Results

This notebook expects the Task 8 result/metrics contract from the canonical pipeline:
- Macro F1 is the primary metric (`primary_metric_name = 'macro_f1'`, `primary_metric`).
- Shared result JSON fields: `metrics`, `split_metadata`, and `artifacts`.
- Canonical metric keys: `macro_f1`, `weighted_f1`, `accuracy`, `precision`, `recall`, `top3_accuracy`, `top5_accuracy`, `per_class`, and `confusion_matrix`.


In [ ]:
from datetime import datetime

eval_results = {
    'model_path': MODEL_PATH,
    'test_accuracy': float(accuracy),
    'per_class_accuracy': {cls: float(acc) for cls, acc in zip(classes, per_class_acc)},
    'confusion_matrix': cm.tolist(),
    'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S')
}

eval_path = os.path.join(WORK_DIR, 'results', f'evaluation_{eval_results["timestamp"]}.json')
with open(eval_path, 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f'Evaluation results saved to: {eval_path}')

## 10. Evaluation Complete

Proceed to inference notebook (04_inference.ipynb)